# Charity Sale Analysis Walkthrough

This notebook gives a short walkthrough of the cleaned charity sale data. It is meant for someone who wants to understand the project without reading every Python script.

## 1. Introduction

The project uses anonymized donation, inventory, sale, and booth planning records from a community charity sale. The data is sample-based and does not include private names, contact information, school names, organization names, addresses, QR codes, or payment details.

In [ ]:
from pathlib import Path
import json
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
processed_dir = project_root / "data" / "processed"
summary_dir = project_root / "reports" / "summary_tables"
metrics_path = project_root / "models" / "model_metrics.json"

## 2. Load Cleaned Data

In [ ]:
donations = pd.read_csv(processed_dir / "cleaned_donations.csv")
inventory = pd.read_csv(processed_dir / "cleaned_inventory.csv")
sales = pd.read_csv(processed_dir / "cleaned_sales.csv")
merged = pd.read_csv(processed_dir / "merged_event_data.csv")
booth_summary = pd.read_csv(summary_dir / "booth_summary.csv")

len(donations), len(inventory), len(sales)

## 3. Donation Summary

In [ ]:
donation_summary = donations.groupby("donor_type", as_index=False)["donation_amount_cny"].sum()
total_direct_donations = donations["donation_amount_cny"].sum()
donation_summary, total_direct_donations

## 4. Inventory Summary

In [ ]:
inventory_summary = inventory.groupby("item_category", as_index=False).agg(
    total_quantity=("quantity", "sum"),
    estimated_total_value_cny=("estimated_total_value_cny", "sum"),
)
inventory_summary.sort_values("total_quantity", ascending=False).head(10)

## 5. Sales Summary

In [ ]:
sales_summary = sales.groupby("item_category", as_index=False).agg(
    quantity_sold=("quantity_sold", "sum"),
    sale_revenue_cny=("total_sale_cny", "sum"),
)
total_sale_revenue = sales["total_sale_cny"].sum()
total_funds = total_direct_donations + total_sale_revenue
sales_summary.sort_values("sale_revenue_cny", ascending=False), total_sale_revenue, total_funds

## 6. Booth Summary

In [ ]:
booth_summary[["booth_area", "assigned_team", "main_category", "actual_items", "booth_revenue_cny"]]

## 7. Estimate vs Actual Price

In [ ]:
estimate_vs_actual = pd.read_csv(summary_dir / "estimate_vs_actual_summary.csv")
estimate_vs_actual[["item_category", "estimated_sold_value_cny", "actual_sale_total_cny", "price_difference_cny"]]

## 8. Simple ML Results

In [ ]:
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
else:
    metrics = {}
metrics

## 9. Key Findings

- Direct donations made up the largest part of the total funds.
- Sale records were useful for reviewing category and booth performance.
- Item IDs made it possible to connect inventory and sale records.
- Estimate vs actual price comparison helped review pricing choices.
- The simple ML results are useful for learning, but the dataset is too small for official decisions.

## 10. Reflection

Before this project, I thought charity sale work was mostly about collecting and selling items. After working on the data side, I learned that clear records, item categories, estimated values, and simple analysis can make an event easier to organize and improve.